# Evaluación con/sin RAG en Colab — genera `eval_resultados.json`

Corre la evaluación (4 artefactos, con vs sin RAG) con **qwen2.5:14b-instruct** (el modelo que
usamos) y **descarga** `data/eval_resultados.json` + la figura `figs/aporte_rag.png`.

**Antes de ejecutar:**
1. 🖥️ *Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU***.
2. ▶️ Ejecutá la celda. Cuando lo pida, subí **`Entrega-Grupo14-GeneradorAcademico.zip`**.
3. ⏳ Paciencia: con 14b son **~8–12 min** (baja el modelo + evalúa). Al terminar se descargan los 2 archivos.

> Para ir más rápido (menor calidad) cambiá `OLLAMA_MODEL` a `qwen2.5:7b-instruct`.
> La celda es autocontenida (no depende de `scripts/`): replica `scripts/run_eval.py`.


In [ ]:
import os, sys, time, json, zipfile, subprocess
from pathlib import Path
from google.colab import files

# 1) Subí el zip de la entrega
up = files.upload()
z = [n for n in up if n.endswith(".zip")][0]
with zipfile.ZipFile(z) as f:
    f.extractall(); root = f.namelist()[0].split("/")[0]
os.chdir(root)

# 2) Dependencias del proyecto
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# 3) Ollama + modelo en la GPU (14b = el que usamos de verdad)
subprocess.run("apt-get -qq install -y zstd pciutils", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_MODEL"] = "qwen2.5:14b-instruct"   # <- "qwen2.5:7b-instruct" para ir más rápido
subprocess.Popen(["ollama", "serve"]); time.sleep(5)
subprocess.run(["ollama", "pull", os.environ["OLLAMA_MODEL"]])

# 4) Bases + evaluación (4 artefactos, con vs sin RAG) -> data/eval_resultados.json
sys.path.insert(0, "src")
import db, ingest, evaluar, generar, figuras
db.reindexar(); ingest.reindexar()

CASOS = [
    ("plan_cursada",           dict(alumno="Simón Ocampo",        tono="tecnico")),
    ("recomendar_orientacion", dict(alumno="Mora Gentil",         tono="tecnico")),
    ("informe_trayectoria",    dict(alumno="Santiago Natalichio", tono="honesto")),
    ("carta_pasantia",         dict(alumno="Lautaro Aubert",      tono="tecnico")),
]
res = []
for nombre, kw in CASOS:
    print("...", nombre, kw, flush=True)
    r = evaluar.comparar_con_sin_rag(getattr(generar, nombre), **kw)
    c, s = r["con_rag"], r["sin_rag"]
    res.append({
        "artefacto": nombre, "args": kw,
        "grounding_con_rag": c["grounding"], "grounding_sin_rag": s["grounding"],
        "grounding_sem_con_rag": c.get("grounding_sem"), "grounding_sem_sin_rag": s.get("grounding_sem"),
        "precision_factual_con_rag": (c.get("precision_factual") or {}).get("precision"),
        "precision_factual_sin_rag": (s.get("precision_factual") or {}).get("precision"),
        "faithfulness_con_rag": c.get("faithfulness"), "faithfulness_sin_rag": s.get("faithfulness"),
        "uso_contexto_con_rag": r.get("uso_contexto_con_rag"), "aporte_contexto": r.get("aporte_contexto"),
        "salida_con_rag": c["salida"], "salida_sin_rag": s["salida"], "contexto": r["contexto"],
    })
    print(f"   uso_contexto={r.get('uso_contexto_con_rag')} aporte_RAG={r.get('aporte_contexto')} "
          f"| lex {c['grounding']}/{s['grounding']} | faith {c.get('faithfulness')}/{s.get('faithfulness')}", flush=True)

Path("data/eval_resultados.json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
figuras.fig_aporte_rag()

# 5) Descargar resultados + figura
files.download("data/eval_resultados.json")
files.download("figs/aporte_rag.png")
